In [ ]:
"""
AgroAlert Ghana -- Extended NDVI Collection (2019-01-01 to 2023-12-31)
=======================================================================
Same logic as original 02_ndvi_data.ipynb (final cell), extended date range.
Source: Sentinel-2 Surface Reflectance (COPERNICUS/S2_SR_HARMONIZED)

NOTE ON DATE RANGE: Sentinel-2 SR_HARMONIZED coverage over Ghana is
consistently available from 2019 onward (launch was 2015/2017, but usable
harmonized SR coverage with acceptable revisit frequency in West Africa is
more reliable from 2019). This matches the "2019 onward" decision.

Run this in the same environment as your original notebooks, with your
service_account.json in place.
"""

import ee
import pandas as pd
import os

# ---- Initialize GEE ----
credentials = ee.ServiceAccountCredentials(
    email='agroalert-service@ee-ishmaelfc.iam.gserviceaccount.com',
    key_file='C:/Users/ELITE/Documents/AGROALERT/service_account.json'
)
ee.Initialize(credentials)

# ---- All 15 communities (unchanged from original) ----
communities = [
    {"name": "Tamale",           "region": "Northern",      "lat": 9.4008,  "lon": -0.8393},
    {"name": "Techiman",         "region": "Bono East",     "lat": 7.5833,  "lon": -1.9333},
    {"name": "Kumasi",           "region": "Ashanti",       "lat": 6.6885,  "lon": -1.6244},
    {"name": "Ho",               "region": "Volta",         "lat": 6.6000,  "lon":  0.4700},
    {"name": "Bolgatanga",       "region": "Upper East",    "lat": 10.7856, "lon": -0.8514},
    {"name": "Wa",               "region": "Upper West",    "lat": 10.0601, "lon": -2.5099},
    {"name": "Sunyani",          "region": "Bono",          "lat": 7.3349,  "lon": -2.3123},
    {"name": "Koforidua",        "region": "Eastern",       "lat": 6.0940,  "lon": -0.2591},
    {"name": "Cape Coast",       "region": "Central",       "lat": 5.1053,  "lon": -1.2466},
    {"name": "Sefwi Wiawso",     "region": "Western North", "lat": 6.2069,  "lon": -2.4856},
    {"name": "Damongo",          "region": "Savannah",      "lat": 9.0833,  "lon": -1.8167},
    {"name": "Nalerigu",         "region": "North East",    "lat": 10.5167, "lon": -0.3667},
    {"name": "Dambai",           "region": "Oti",           "lat": 8.0667,  "lon":  0.1833},
    {"name": "Goaso",            "region": "Ahafo",         "lat": 6.8017,  "lon": -2.5181},
    {"name": "Sekondi-Takoradi", "region": "Western",       "lat": 4.9347,  "lon": -1.7137},
]

# ---- EXTENDED DATE RANGE ----
START_DATE = '2019-01-01'
END_DATE = '2023-12-31'


def get_ndvi(community, start_date, end_date):
    point = ee.Geometry.Point([community['lon'], community['lat']])
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(point)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(lambda img: img.normalizedDifference(['B8', 'B4'])
             .rename('NDVI')
             .set('date', img.date().format('YYYY-MM-dd'))))

    def extract(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point.buffer(5000),
            scale=100
        )
        return ee.Feature(None, {
            'date': img.get('date'),
            'ndvi': val.get('NDVI'),
            'community': community['name'],
            'region': community['region']
        })

    return collection.map(extract)


print(f"Pulling NDVI for {len(communities)} communities, {START_DATE} to {END_DATE}")
print("This will take considerably longer than the 2022-2023 pull (~2.5x the date range).")
print("Consider running per-community if you hit GEE timeout/quota limits (see notes below).\n")

all_records = []

# NOTE: Pulling all 15 communities x 5 years in one .getInfo() call risks
# GEE request timeouts. This version pulls COMMUNITY-BY-COMMUNITY and
# concatenates locally, which is more robust for a longer date range.
for i, community in enumerate(communities):
    print(f"  [{i+1}/{len(communities)}] Fetching {community['name']}...")
    fc = get_ndvi(community, START_DATE, END_DATE)
    try:
        data = fc.getInfo()
    except Exception as e:
        print(f"    ERROR fetching {community['name']}: {e}")
        print(f"    Skipping -- you may need to retry this community alone, or split into year chunks.")
        continue

    for feature in data['features']:
        props = feature['properties']
        all_records.append({
            'community': props.get('community'),
            'region': props.get('region'),
            'date': props.get('date'),
            'ndvi': props.get('ndvi')
        })
    print(f"    -> {len(data['features'])} records")

df_ndvi = pd.DataFrame(all_records)
df_ndvi['date'] = pd.to_datetime(df_ndvi['date'])
df_ndvi = df_ndvi.sort_values(['community', 'date']).reset_index(drop=True)
df_ndvi = df_ndvi.dropna(subset=['ndvi'])

os.makedirs('C:/Users/ELITE/Documents/AGROALERT/data_raw', exist_ok=True)
OUTPUT_PATH = 'C:/Users/ELITE/Documents/AGROALERT/data_raw/ndvi_raw_2019_2023.csv'
df_ndvi.to_csv(OUTPUT_PATH, index=False)

print(f"\nDone. Records saved: {len(df_ndvi)}")
print(f"Saved to: {OUTPUT_PATH}")
print(f"Communities covered: {df_ndvi['community'].nunique()} / {len(communities)}")
print(f"Date range: {df_ndvi['date'].min()} to {df_ndvi['date'].max()}")
print(f"\nNDVI range: {df_ndvi['ndvi'].min():.3f} to {df_ndvi['ndvi'].max():.3f}")

print("\n--- IMPORTANT ---")
print("Once you've verified this file, you'll want to MERGE it with the original")
print("2022-2023 ndvi_raw.csv OR simply replace it, since this pull already spans")
print("the full 2019-2023 window. Recommend: replace, don't merge, to avoid duplicate rows.")
